In [9]:
import pandas as pd
import configPinecone
import pinecone
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer as ST

In [2]:
files = pd.read_csv("course_descriptions.csv", encoding = "ANSI") #default utf-8

In [3]:
def createCourseDescription(row): # longer text with embedded context
    return f"""The course name is {row["course_name"]}, the slug is {row["course_slug"]}, 
    the technology is {row["course_technology"]} and the course topic is {row["course_topic"]}."""

In [4]:
files["course_description_new"] = files.apply(createCourseDescription, axis = 1)
files.head(5)

,course_name,course_slug,course_technology,course_description,course_topic,course_description_short,course_description_new
0,Introduction to Tableau,tableau,tableau,Tableau is now one of the most popular busines...,data visualization,Teaching you how to tell compelling stories wi...,"The course name is Introduction to Tableau, th..."
1,The Complete Data Visualization Course with Py...,data-visualization,python,The Data Visualization course is designed for ...,data visualization,Teaching you how to master the art of creating...,The course name is The Complete Data Visualiza...
2,Introduction to R Programming,introduction-to-r-programming,r,R is one of the best programming languages spe...,programming,"Providing you with the skills to manipulate, a...",The course name is Introduction to R Programmi...
3,Data Preprocessing with NumPy,data-preprocessing-numpy,python,This course is designed to show you how to wor...,data processing,This course will guide you through one of Pyth...,The course name is Data Preprocessing with Num...
4,Introduction to Data and Data Science,intro-to-data-and-data-science,theory,Working with data is an essential part of main...,machine learning,Introducing you to the field of data science a...,The course name is Introduction to Data and Da...


In [5]:
pc = Pinecone(api_key = configPinecone.pineconeDefaultAPIKey, environment = configPinecone.pineconeEnv)

In [6]:
indexN = "semantic-search-index"
dimension = 384
metric = "cosine"

In [7]:
if indexN in [index.name for index in pc.list_indexes()]:
    pc.delete(indexN)
pc.create_index(
    name = indexN,
    dimension = dimension,
    metric = metric,
    spec = ServerlessSpec(
        cloud = "aws",
        region = "us-east-1"
    )
)

{
    "name": "semantic-search-index",
    "metric": "cosine",
    "host": "semantic-search-index-cwx78x5.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "vary": "origin, access-control-request-method, access-control-request-headers",
            "access-control-allow-origin": "*",
            "access-control-expose-headers": "*",
            "x-pinecone

In [8]:
index = pc.Index(indexN)

In [11]:
model = ST("all-MiniLM-L6-v2")

In [15]:
def createEmbeddings(row):
    combinedTxt = " ".join([str(row[field]) for field in ["course_description", "course_description_new", "course_description_short"]])
    embedding = model.encode(combinedTxt, show_progress_bar = False)
    return embedding

In [16]:
files["embedding"] = files.apply(createEmbeddings, axis = 1)

In [17]:
vectorsToUpsert = [(str(row["course_name"]), row["embedding"].tolist()) for _, row in files.iterrows()]
index.upsert(vectorsToUpsert)
print("Data upserted to Pinecone index")

Data upserted to Pinecone index


In [18]:
query = "clustering"
queryEmbedding = model.encode(query, show_progress_bar = False).tolist() #numpy array, expected format for pinecone

In [19]:
queryResults = index.query(
    vector = [queryEmbedding], # pinecone expects multiple vectors
    top_k = 10, # starting with the closest match
    include_values = True    
)

In [20]:
queryResults

QueryResponse(matches=[{'id': 'Machine Learning in Excel',
 'score': 0.360274315,
 'values': [-0.0137635227,
            -0.0262684152,
            -0.0204973333,
            -0.0147366812,
            -0.0264848601,
            -0.0213876124,
            -0.0543333218,
            -0.0524257943,
            0.00822975,
            0.0254937187,
            -0.0382547118,
            -0.040978916,
            0.0701174363,
            -0.0389499255,
            -0.0068457569,
            0.0402314775,
            -0.00301266275,
            -0.00384135242,
            -0.00449051382,
            -0.0617037378,
            0.0832237229,
            0.0293510389,
            -0.0514961593,
            -0.0214260556,
            0.0418344364,
            0.0307958815,
            0.06357117,
            0.00410186872,
            -0.0458159819,
            -0.0352171548,
            -0.0466994978,
            0.00587529782,
            0.0160002597,
            0.0544324815,
            -

In [28]:
for match in queryResults["matches"]:
    print(f"""Matched item ID: {match["id"]}, score: {match["score"]}""")

Matched item ID: Machine Learning in Excel, score: 0.360274315
Matched item ID: Machine Learning with K-Nearest Neighbors, score: 0.317268372
Matched item ID: Customer Churn Analysis with SQL and Tableau, score: 0.276978016
Matched item ID: Machine Learning in Python, score: 0.268226653
Matched item ID: Linear Algebra and Feature Selection, score: 0.256421119
Matched item ID: Growth Analysis with SQL, Python, and Tableau  , score: 0.254556656
Matched item ID: Fashion Analytics with Tableau, score: 0.235390663
Matched item ID: Customer Engagement Analysis with SQL and Tableau, score: 0.235216141
Matched item ID: Machine Learning with Support Vector Machines, score: 0.226407051
Matched item ID: Machine Learning with Naive Bayes, score: 0.223970428


In [29]:
files = pd.read_csv("course_section_descriptions.csv", encoding = "ANSI")
files["unique_id"] = files["course_id"].astype(str) + "-" + files["section_id"].astype(str)
files["metadata"] = files.apply(lambda row: {
    "course_name" : row["course_name"],
    "section_name" : row["section_name"],
    "section_description" : row["section_description"],
}, axis = 1)

In [30]:
def createEmbeddings(row):
    combinedTxt = f"""
    {row["course_name"]} {row["course_technology"]} {row["course_description"]} {row["section_name"]} {row["section_description"]}
    """
    return model.encode(combinedTxt, show_progress_bar = False)

In [33]:
files["embedding"] = files.apply(createEmbeddings, axis = 1)

In [34]:
vectorsToUpsert = [(row["unique_id"], row["embedding"].tolist(), row["metadata"]) for idx, row in files.iterrows()]

In [43]:
index.upsert(vectors = vectorsToUpsert)
print("Data upserted to pinecone index")

Data upserted to pinecone index
